# Intro to PandaPower for IDP Project Optimization

In [3]:
try:
    from pandapower.networks.power_system_test_cases import case30, case_ieee30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries, OutputWriter
    from pandapower.diagnostic import Diagnostic
    from pandapower.create import create_storage
    from tools.limits import bus_vm_pu_limits, line_loading_limits
    from tools.jacobian import voltage_sensitivity_matrix, vs_mse   

except: 
    %pip install pandapower
    %pip install numba
    %pip install matplotlib
    %pip install scikit-learn
    from pandapower.networks.power_system_test_cases import case30, case_ieee30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries
    from tools.jacobian import voltage_sensitivity_matrix, vs_mse   

from control.battery import Battery
import profileloader as pl
import numpy as np
import os
import tempfile

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 

By default, the IEEE 30 net has the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [4]:
net = case30()
print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


In [ ]:
net.bus


,name,vn_kv,type,zone,in_service,max_vm_pu,min_vm_pu,geo
0,1,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [-1.7274567146, -3.1892649529]..."
1,2,135.0,b,1.0,True,1.10,0.95,"{""coordinates"": [-0.5620171842, -3.3148524773]..."
2,3,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [-1.59061066, -4.144202153], ""..."
3,4,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [-0.3726946517, -4.2829075502]..."
4,5,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [-1.0537836171, -2.2394463492]..."
5,6,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [0.5309084006, -3.433272374], ..."
6,7,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [-0.0673791279, -2.3918873148]..."
7,8,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [1.0669150553, -2.4311581116],..."
8,9,135.0,b,1.0,True,1.05,0.95,"{""coordinates"": [1.1037817517, -3.1875438028],..."
9,10,135.0,b,3.0,True,1.05,0.95,"{""coordinates"": [1.1626994941, -4.2978495824],..."


In [ ]:
store_el = create_storage( 
    net, 21, 
    p_mw = 20, 
    q_mvar = 0, 
    controllable=True,
    max_e_mwh = 120, 
    soc_percent=50,
    max_p_mw=60, 
    min_p_mw=-60, 
    max_q_mvar=30,
    min_q_mvar=-30)

print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - storage (2 elements)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


In [ ]:
store_el.item()

1

Specific cells are adjustable based on external data

In [ ]:
# for example: set the power consumption of load #1 to 2x its starting value
"""print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 2 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

# and: set line loading limit to 50%
print("Before: " + str(net.line.at[10,'max_loading_percent']))
net.line.at[10,'max_loading_percent'] = 0.5
print("After: " + str(net.line.at[10,'max_loading_percent']))"""

# and: set generator 5 to limit to 2x its starting value
print("Before: " + str(net.gen.at[1,'p_mw']))
net.gen.at[4,'p_mw'] = 3 * net.gen.iloc[4].p_mw
print("After: " + str(net.gen.at[4,'p_mw']))

# and: set generator 5 to limit to 2x its starting value
print("Before: " + str(net.gen.at[3,'p_mw']))
net.gen.at[3,'p_mw'] = 3 * net.gen.iloc[3].p_mw
print("After: " + str(net.gen.at[3,'p_mw']))

### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [ ]:
runpp(net)
vs_mse(net)

You can use the result dataframes to check if power flow data is within the defined limits for each object

In [ ]:
# For example, buses:
bus_vm_pu_limits(net)
line_loading_limits(net)

For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [ ]:
try: 
    runopp(net)
except: 
    diag = Diagnostic()
    result = diag.diagnose_network(net, report_style="detailed")    


Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [ ]:
print(net.gen.head())

In [ ]:
# proof that bus max/min values are now accounted for
bus_vm_pu_limits(net)
line_loading_limits(net)

To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [ ]:
net.gen.at[4,'p_mw'] = 20
net.gen.at[4, 'controllable'] = False

In [ ]:
runpp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

In [ ]:
runopp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

The resulting optimized components have changed based on this new constraint. 

### Analysis with Custom Data

Load in custom loads/gens from the provided Excel sheets

In [ ]:
net=case30()

net.poly_cost["cp2_eur_per_mw2"] = 0        # clear out default costs
"""store_el = create_storage( 
    net, 5, 
    p_mw = 0, 
    q_mvar = 0, 
    controllable=True,
    max_e_mwh = 40, 
    soc_percent=60,
    max_p_mw=10, 
    min_p_mw=-10, 
    max_q_mvar=10,
    min_q_mvar=-10)"""


load_control = pl.json_to_net(net, "load", date="1/2/2026")
gen_control = pl.json_to_net(net, "gen", date="1/2/2026")
tariff_control = pl.json_to_net(net, "poly_cost", date="1/2/2026")
#storage_control = Battery(net=net, element_index=0)

In [7]:
for _, ctrl in net.controller.iterrows():
    print(ctrl.object.element)
    print(ctrl.object.element_index)
    el = ctrl.object.element
    idx = ctrl.object.element_index

    print(net[el].index)

load
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17], dtype='int64')
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17], dtype='int64')
storage
Index([0, 1], dtype='int64')
Index([0, 1], dtype='int64')
gen
Index([0], dtype='int64')
Index([0], dtype='int64')
poly_cost
Index([0, 1, 2, 3, 4, 5], dtype='int64')
Index([0, 1, 2, 3, 4, 5], dtype='int64')


In [ ]:
def run_test(net, **kwargs):
    runpp(net=net, **kwargs)
    bus_vm_pu_limits(net)
    line_loading_limits(net)

ow = OutputWriter(net, 
                  output_path="..\\results", 
                  output_file_type='.csv', csv_separator=",")
ow.log_variable("res_storage", "p_mw")
ow.log_variable("storage", "soc_percent")
run_timeseries(net, continue_on_divergence=True
               , max_iteration=40, verbose=True, run=run_test)

In [13]:
import profileloader as pl
net = case30()
test_date = "5/8/2026"

# import profiles for timeseries data
load_control = pl.json_to_net_generic(net, "load", date=test_date)
hydro_control = pl.json_to_net_hydro(net, date=test_date)
gen_control = pl.json_to_net_pv(net, date=test_date)
tariff_control = pl.json_to_net_generic(net, "poly_cost", date=test_date)

In [16]:
for i, row in net.controller.iterrows():
    ctrl = row.object

    print("\nController:", i)
    print("profile_name:", ctrl.profile_name)

    try:
        print("df columns:", ctrl.data_source.df.columns)
        print("column types:",
              [type(c) for c in ctrl.data_source.df.columns])

        test = ctrl.data_source.df.loc[0, ctrl.profile_name]
        print("OK")

    except Exception as e:
        print("FAILED:", e)


Controller: 0
profile_name: Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17], dtype='int64')
df columns: Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], dtype='int64')
column types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
OK

Controller: 1
profile_name: Index([0, 1], dtype='int64')
df columns: Index([0, 1], dtype='int64')
column types: [<class 'int'>, <class 'int'>]
OK

Controller: 2
profile_name: Index([0], dtype='int64')
df columns: Index([0, 1, 2, 3, 4], dtype='int64')
column types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
OK

Controller: 3
profile_name: Index([0, 1, 2, 3, 4, 5], dtype='int64')
df columns: Index([0, 1, 2, 3, 4, 5], dtype='int

In [8]:
def run_test(net, **kwargs):
    runpp(net=net, **kwargs)
    bus_vm_pu_limits(net)
    line_loading_limits(net)

ow = OutputWriter(net, 
                  output_path="..\\results", 
                  output_file_type='.csv', csv_separator=",")
ow.log_variable("res_storage", "p_mw")
ow.log_variable("storage", "soc_percent")
run_timeseries(net, continue_on_divergence=True, max_iteration=40, verbose=True, run=run_test)


No time steps to calculate are specified. I'll check the datasource of the first controller for avaiable time steps

  0%|          | 0/96 [00:00<?, ?it/s]

KeyError: '[0] not in index'

In [ ]:
net.con

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,max_loading_percent,geo
0,None,None,0,1,1.0,3.6450,10.9350,436.639076,0.0,0.555967,1.0,1,ol,True,100.0,None
1,None,None,0,2,1.0,9.1125,34.6275,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None
2,None,None,1,3,1.0,10.9350,30.9825,291.092717,0.0,0.277983,1.0,1,ol,True,100.0,None
3,None,None,2,3,1.0,1.8225,7.2900,0.000000,0.0,0.555967,1.0,1,ol,True,100.0,None
4,None,None,1,4,1.0,9.1125,36.4500,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None
5,None,None,1,5,1.0,10.9350,32.8050,291.092717,0.0,0.277983,1.0,1,ol,True,100.0,None
6,None,None,3,5,1.0,1.8225,7.2900,0.000000,0.0,0.384900,1.0,1,ol,True,100.0,None
7,None,None,4,6,1.0,9.1125,21.8700,145.546359,0.0,0.299367,1.0,1,ol,True,100.0,None
8,None,None,5,6,1.0,5.4675,14.5800,145.546359,0.0,0.555967,1.0,1,ol,True,100.0,None
9,None,None,5,7,1.0,1.8225,7.2900,0.000000,0.0,0.136853,1.0,1,ol,True,100.0,None
